In [1]:
import os
import urllib.request

In [2]:
with open('../../dataset/the-verdict.txt', 'r') as f:
    text_data = f.read()
print(len(text_data))

20479


In [3]:
#Tokenization Simple
import re
text = "Hello, world. This--, is a test."
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result

result = [x.strip() for x in result if x.strip()]
result

['Hello', ',', 'world', '.', 'This', '--', ',', 'is', 'a', 'test', '.']

In [4]:
#Tokenization 
import re
result = re.split(r'([,.:;?_!"()\']|--|\s)', text_data)
result

preprocessed_data = [x.strip() for x in result if x.strip()]
len(preprocessed_data)

4690

In [5]:
# Convert Tokens to Token Id's

vocabulary = {token: token_id for token_id, token in enumerate(sorted(set(preprocessed_data)))}
len(vocabulary)
vocabulary

token_data = []
for word in preprocessed_data:
    token_data.append(vocabulary.get(word))
len(token_data)

4690

In [6]:
class MyTokenizer():
    def __init__(self, vocabulary):
        self.string_to_int = vocabulary
        self.int_to_String = {token_id: token for token, token_id in vocabulary.items()}

    def encode(self, text_data):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text_data)
        preprocessed = [x.strip() for x in preprocessed if x.strip()]

        preprocessed = [word if word in self.string_to_int else '|unk|' for word in preprocessed]
        token_ids = [self.string_to_int.get(word) for word in preprocessed]
        return token_ids
    
    def decode(self, token_data):
        text = ' '.join([self.int_to_String.get(token) for token in token_data])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

# Adding special context tokens
all_tokens = sorted(list(set(preprocessed_data)))
all_tokens.extend(['<|endoftext|>', '|unk|'])
vocabulary = {token: token_id for token_id, token in enumerate(all_tokens)}


tokenizer = MyTokenizer(vocabulary)
encoded_data = tokenizer.encode('Hello, I HAD always thought Jack Gisburn rather')
print(encoded_data)
tokenizer.decode(encoded_data)




[1131, 5, 53, 44, 149, 1003, 57, 38, 818]


'|unk|, I HAD always thought Jack Gisburn rather'

In [7]:
# Byte pair encoding - Tokenizer - GPT2 library implemented in RUST 0.9.0
import tiktoken
tiktoken.__version__

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode("hello, how are you ewe <|endoftext|>", allowed_special={'<|endoftext|>'})

tokenizer.n_vocab
encoded

[31373, 11, 703, 389, 345, 304, 732, 220, 50256]

In [8]:
# Data sampling with sliding window

with open('../../dataset/the-verdict.txt', 'r') as f:
    text_data = f.read()

encoded_token = tokenizer.encode(text_data)
print(encoded_token[:10])

context_size = 4

for i in range(1, context_size):
    x = encoded_token[:i]
    y = encoded_token[i]
    print(f"{x} --> {y}")




[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
[40] --> 367
[40, 367] --> 2885
[40, 367, 2885] --> 1464


In [9]:
# using pytorch 2.6.0
import torch
import torch.nn as nn
torch.__version__
from torch.utils.data import Dataset, DataLoader

class CustomDataSet(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(text, allowed_special={'<|endoftext|>'})

        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i : i +max_length]
            target_chunk = token_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.output_ids[index]

In [10]:
def createDataLoader(text, batch_size=4, max_length=256, stride=128, shuffle= True,
                     drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = CustomDataSet(text_data, tokenizer, max_length, stride)

    dataloader = DataLoader(
                        dataset=dataset, 
                        batch_size=batch_size, 
                        shuffle=shuffle,
                        drop_last=drop_last,
                        num_workers=num_workers
                           )
    return dataloader

In [11]:
#stride of 4 is used so that each time diff window and which reduces overfitting
data_loader = createDataLoader(text_data, batch_size=8, 
                               max_length=4, stride=4, shuffle=False)
next(data_loader._get_iterator())

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

In [12]:
# creating token embeddings - small test 
# initally the embedding metrix (linear layer) are initailized with random weights
# and then gets correct weight while in training

context_size = 4
vocab_size = tokenizer.n_vocab
output_dim = 256
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)
#print(embedding_layer.weight)
#embedding_layer(torch.tensor([0]))

position_embedding_layer = nn.Embedding(num_embeddings=context_size, embedding_dim=output_dim)

data_loader = createDataLoader(text_data, batch_size=8, 
                               max_length=context_size, stride=4, shuffle=False)
for input_feature, target in data_loader:
    #print(f"input_feature: {input_feature}")
    token_embeddings = embedding_layer(input_feature) + position_embedding_layer(torch.arange(context_size))
    
token_embeddings  

tensor([[[ 0.2752, -1.1898, -1.0993,  ...,  0.0761, -0.4461, -0.8240],
         [-0.8941,  0.0991, -0.4763,  ..., -0.8183,  0.8952,  0.6413],
         [ 0.5402,  0.1440,  0.0593,  ...,  0.3508, -0.4800,  0.3488],
         [-1.0790,  1.0122, -1.5857,  ..., -1.2790,  0.3808, -0.4785]],

        [[-2.2669,  0.5449, -2.5579,  ...,  2.3268,  0.4769, -0.5701],
         [-1.6346,  0.6561, -2.3923,  ...,  0.3456, -0.5987, -0.1568],
         [ 1.4199,  1.9513,  1.0927,  ...,  0.8226,  0.3353,  0.1593],
         [ 0.9655,  1.8710, -2.2451,  ..., -1.0703, -1.7070,  0.8288]],

        [[-0.5468, -1.5333,  0.0148,  ...,  1.8072,  0.1700, -1.2257],
         [-0.6873,  1.3759,  0.5021,  ...,  1.5432, -3.2914,  1.4845],
         [ 0.2283,  0.8336, -0.7951,  ...,  0.7220, -0.9117, -0.2881],
         [ 1.9546,  0.7150,  0.0098,  ..., -1.5228, -0.3245,  0.7943]],

        ...,

        [[-0.1916,  1.8306, -3.4028,  ..., -0.1392, -0.0364, -2.7723],
         [-0.0286,  1.0850, -0.2893,  ...,  1.7414,  1.73

In [13]:
# Positional encoding - also initally random and  learned while training

position_embedding_layer

Embedding(4, 256)

In [14]:
# add token embeddings and positional embeddings

position_embedding_layer = nn.Embedding(num_embeddings=context_size, embedding_dim=output_dim)
position_embedding_layer.weight.shape
# added pos embedding above cells


torch.Size([4, 256])